# Step 3 — Dopamine neuron subtypes and senescence scoring (GSE178265)

**Project:** pd-da-aging · **Follows:** docs/ANALYSIS_PLAN.md (pre-registered before any scoring)

Run the cells **one at a time, from top to bottom** (click ▶ on each). Stop at each 🛑 checkpoint and send Claude a screenshot.

**Needs:** `GSE178265_DA_neurons.h5ad` in *My Drive* (made in step 2). **Results go to:** *My Drive › pd-da-aging_step3*.

## Cell 1 — Connect Google Drive and install tools
Click **Allow** when Google asks to connect your Drive. Takes 2–3 minutes.

In [ ]:
# Connect Google Drive and install tools (2-3 minutes)
from google.colab import drive
drive.mount("/content/drive")
!pip install -q scanpy harmonypy igraph leidenalg statsmodels openpyxl gseapy


## Cell 2 — Record software versions
Saves a small text file so reviewers know exactly which software was used.

In [ ]:
# Record software versions and set up the output folder (seconds)
import os, sys, json, re, io, zipfile, datetime, platform, urllib.request
import numpy as np, pandas as pd, scipy, scanpy as sc, anndata as ad
import statsmodels, statsmodels.formula.api as smf
from scipy import stats, sparse
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*(converge|singular|boundary|Hessian).*")

DRIVE = "/content/drive/MyDrive"
OUT = f"{DRIVE}/pd-da-aging_step3"
for sub in ["", "/figures", "/tables", "/gene_sets"]:
    os.makedirs(OUT + sub, exist_ok=True)
TODAY = datetime.date.today().isoformat()

import harmonypy, igraph, leidenalg
versions = {"date": TODAY, "python": sys.version.split()[0], "platform": platform.platform()}
for m in [np, pd, scipy, sc, ad, statsmodels, matplotlib, harmonypy, igraph, leidenalg]:
    versions[m.__name__] = getattr(m, "__version__", "unknown")
with open(f"{OUT}/software_versions.txt", "w") as fh:
    fh.write("\n".join(f"{k}: {v}" for k, v in versions.items()) + "\n")
print("\n".join(f"{k}: {v}" for k, v in versions.items()))
print(f"\nResults will be saved in My Drive > pd-da-aging_step3")


## Cell 3 — Load the dopamine neurons from step 2
Should say about **16,755 neurons** from **18 donors**.

In [ ]:
# Load the 16,755 dopamine neurons from step 2 (1-2 minutes)
da = ad.read_h5ad(f"{DRIVE}/GSE178265_DA_neurons.h5ad")
da.obs["age"] = pd.to_numeric(da.obs["age"], errors="coerce")
for col in ["donor", "status", "disease", "sex", "facs", "sample"]:
    da.obs[col] = da.obs[col].astype(str)
da.obs["n_genes_raw"] = np.asarray((da.X > 0).sum(axis=1)).ravel()
da.obs["total_counts_raw"] = np.asarray(da.X.sum(axis=1)).ravel()
da.layers["counts"] = da.X.copy()
print(f"{da.n_obs:,} neurons x {da.n_vars:,} genes from {da.obs['donor'].nunique()} donors")
print(da.obs.groupby("status")["donor"].nunique().rename("donors").to_string())


## Cell 4 — Download the gene lists from their original sources
Each list is downloaded from the original paper or database and saved with today's date.

### 🛑 CHECKPOINT 1 — stop here and send Claude a screenshot of this cell's output.

In [ ]:
# Download every gene list from its original source and save it with today's date (1-2 minutes)
def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=120).read()

sources, gene_sets = {}, {}

# 4a. Kamath et al. 2022 DA subtype marker panel: the authors' own Figure 1E code (pinned commit)
KAMATH_URL = ("https://raw.githubusercontent.com/tkamath1/Kamathetal2022/"
              "cae72e19290efe1ea8756e2032507905bdea52ba/Main_Figs/Figure1.R")
fig1 = fetch(KAMATH_URL).decode()
block = re.search(r"DotPlot\(sn\.da\.seurat,genes\.plot = c\((.*?)\),x\.lab\.rot", fig1, re.S).group(1)
kamath_panel = re.findall(r"['\"]([A-Za-z0-9\-]+)['\"]", block)
order_block = re.search(r"levels = c\(('SOX6_AGTR1'.*?)\)\)", fig1, re.S).group(1)
kamath_subtypes = re.findall(r"'([A-Z0-9_]+)'", order_block)
gene_sets["kamath_marker_panel"] = kamath_panel
sources["kamath_marker_panel"] = KAMATH_URL + " (Figure 1E DotPlot genes)"
print(f"Kamath marker panel ({len(kamath_panel)} genes): {kamath_panel}")
print(f"Kamath DA subtypes ({len(kamath_subtypes)}): {kamath_subtypes}")

# 4b. SenMayo (Saul et al. 2022, Nat Commun), Supplementary Data 1
SENMAYO_URLS = [
    "https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-022-32552-1/MediaObjects/41467_2022_32552_MOESM4_ESM.xlsx",
    "https://pmc.ncbi.nlm.nih.gov/articles/instance/9381717/bin/41467_2022_32552_MOESM4_ESM.xlsx",
]
senmayo, senmayo_note = None, ""
for u in SENMAYO_URLS:
    try:
        sheets = pd.read_excel(io.BytesIO(fetch(u)), sheet_name=None, header=None)
        sources["SenMayo"] = u
        break
    except Exception as e:
        senmayo_note += f"{u}: {e}\n"
if "SenMayo" in sources:
    # Find the column of human gene symbols: the column whose header mentions 'human',
    # otherwise the first column made mostly of upper-case gene-like symbols.
    best = None
    for name, df in sheets.items():
        for j in range(df.shape[1]):
            col = df.iloc[:, j].dropna().astype(str).str.strip()
            header_text = " ".join(col.head(3)).lower()
            sym = col[col.str.fullmatch(r"[A-Z][A-Z0-9\-\.]{1,14}")]
            if len(sym) >= 100 and (best is None or "human" in header_text):
                best = (name, j, sym.tolist())
    if best:
        col = sheets[best[0]].iloc[:, best[1]].dropna().astype(str).str.strip()
        col = col[~col.str.contains(r"gene|human|symbol|\s", case=False, regex=True)]
        senmayo = list(dict.fromkeys(col.tolist()))
        senmayo_note = f"sheet '{best[0]}', column {best[1] + 1}"
gene_sets["SenMayo"] = senmayo
print(f"\nSenMayo: {len(senmayo) if senmayo else 'NOT FOUND'} genes ({senmayo_note.strip()})")

# 4c. CellAge Build 3 (genes whose manipulation induces senescence)
CELLAGE_URL = "https://www.genomics.senescence.info/cells/cellAge.zip"
try:
    z = zipfile.ZipFile(io.BytesIO(fetch(CELLAGE_URL)))
    member = [m for m in z.namelist() if m.lower().endswith((".csv", ".tsv", ".txt"))][0]
    raw = z.read(member).decode("utf-8", errors="replace")
    sep = "\t" if raw.count("\t") > raw.count(";") and raw.count("\t") > raw.count(",") else (";" if raw.count(";") > raw.count(",") else ",")
    cellage = pd.read_csv(io.StringIO(raw), sep=sep)
    print(f"\nCellAge file '{member}' columns: {list(cellage.columns)}")
    gcol = ([c for c in cellage.columns if re.search(r"symbol", c, re.I)] +
            [c for c in cellage.columns if re.search(r"gene.?name", c, re.I)])[0]
    ecol = [c for c in cellage.columns if re.search(r"effect", c, re.I)][0]
    print(f"Effect values: {cellage[ecol].value_counts().to_dict()}")
    gene_sets["CellAge_induces"] = sorted(set(cellage.loc[cellage[ecol].astype(str).str.contains("induc", case=False), gcol].astype(str).str.strip()))
    sources["CellAge_induces"] = f"{CELLAGE_URL} ({member}; '{ecol}' contains 'induces')"
    print(f"CellAge senescence-inducing genes: {len(gene_sets['CellAge_induces'])}")
except Exception as e:
    gene_sets["CellAge_induces"] = None
    print(f"\nCellAge could not be read: {e}")

# 4d. MSigDB C2 (CGP): human brain-aging set (Lu et al. 2004) and the MSigDB copy of SenMayo
msig = {}
for ver in ["2025.1.Hs", "2024.1.Hs", "2023.2.Hs", "2023.1.Hs"]:
    u = f"https://data.broadinstitute.org/gsea-msigdb/msigdb/release/{ver}/c2.cgp.v{ver}.symbols.gmt"
    try:
        for line in fetch(u).decode().splitlines():
            parts = line.split("\t")
            msig[parts[0]] = parts[2:]
        sources["MSigDB_C2_CGP"] = u
        break
    except Exception as e:
        print(f"MSigDB {ver} not available: {e}")
gene_sets["LU_AGING_BRAIN_UP"] = msig.get("LU_AGING_BRAIN_UP")
sources["LU_AGING_BRAIN_UP"] = sources.get("MSigDB_C2_CGP", "") + " (LU_AGING_BRAIN_UP)"
print(f"\nBrain aging (LU_AGING_BRAIN_UP): {len(gene_sets['LU_AGING_BRAIN_UP']) if gene_sets['LU_AGING_BRAIN_UP'] else 'NOT FOUND'} genes")
if senmayo and "SAUL_SEN_MAYO" in msig:
    m = set(msig["SAUL_SEN_MAYO"]); s = set(senmayo)
    print(f"Cross-check: SenMayo supplementary vs MSigDB SAUL_SEN_MAYO: {len(s & m)} shared, "
          f"{len(s - m)} only in supplementary, {len(m - s)} only in MSigDB")

# 4e. Single genes (fixed in the analysis plan)
gene_sets["CDKN1A"] = ["CDKN1A"]
gene_sets["CDKN2A"] = ["CDKN2A"]

with open(f"{OUT}/gene_sets/gene_sets_{TODAY}.json", "w") as fh:
    json.dump({"downloaded": TODAY, "sources": sources, "gene_sets": gene_sets}, fh, indent=1)
print("\nPresent in our data:")
for k, v in gene_sets.items():
    if v:
        print(f"  {k}: {sum(g in da.var_names for g in v)} of {len(v)}")
print(f"\nSaved to My Drive > pd-da-aging_step3 > gene_sets")


## Cell 5 — Re-cluster the dopamine neurons on their own
The senescence and aging genes are kept out of this step, so the subtype labels cannot be influenced by the scores we test later. Takes 5–10 minutes.

In [ ]:
# Re-cluster the dopamine neurons on their own (5-10 minutes)
import numba
sc.settings.n_jobs = max(1, int(numba.config.NUMBA_NUM_THREADS))   # match Colab's CPU count
da.X = da.layers["counts"].copy()       # always start from raw counts, so this cell is safe to re-run
da.uns.pop("log1p", None)               # remove a leftover flag from step 2
sc.pp.normalize_total(da, target_sum=1e4)
sc.pp.log1p(da)
da.layers["lognorm"] = da.X.copy()
sc.pp.highly_variable_genes(da, n_top_genes=2000)
# Keep the senescence/aging readout genes OUT of the clustering, so subtype labels cannot be driven by the scores we test
readout_genes = set().union(*[set(gene_sets[k]) for k in ["SenMayo", "CellAge_induces", "LU_AGING_BRAIN_UP", "CDKN1A", "CDKN2A"] if gene_sets.get(k)])
dropped = int((da.var["highly_variable"] & da.var_names.isin(readout_genes)).sum())
da.var.loc[da.var_names.isin(readout_genes), "highly_variable"] = False
print(f"Variable genes used for clustering: {int(da.var['highly_variable'].sum())} ({dropped} readout genes removed)")
sc.tl.pca(da, n_comps=30, mask_var="highly_variable")
ho = harmonypy.run_harmony(da.obsm["X_pca"], da.obs, "sample", max_iter_harmony=20)
Z = np.asarray(ho.Z_corr)
da.obsm["X_pca_harmony"] = Z if Z.shape[0] == da.n_obs else Z.T
sc.pp.neighbors(da, use_rep="X_pca_harmony", n_neighbors=20)
RESOLUTION = 1.0   # only change this if Claude asks you to at Checkpoint 2
sc.tl.leiden(da, resolution=RESOLUTION, flavor="igraph", n_iterations=2, directed=False, key_added="subcluster")
sc.tl.umap(da)
print(f"{da.obs['subcluster'].nunique()} sub-clusters among {da.n_obs:,} dopamine neurons (resolution {RESOLUTION})")


## Cell 6 — Name each sub-cluster with Kamath's subtype markers
### 🛑 CHECKPOINT 2 — stop here and send Claude screenshots of the table, the counts, the cross-check table and the dot plot.
Do **not** run Cell 7 until Claude confirms the labels look right.

In [ ]:
# Name each sub-cluster by Kamath's subtype markers (seconds)
# Each subtype name is SOX6 or CALB1 plus its specific marker(s): e.g. CALB1_CRYM_CCDC68 = CALB1 + CRYM + CCDC68.
subtype_genes = {s: s.split("_") for s in kamath_subtypes}
missing = sorted({g for gs in subtype_genes.values() for g in gs} - set(da.var_names))
if missing:
    print(f"Note: marker genes not in the data (dropped): {missing}")
panel = [g for g in dict.fromkeys(kamath_panel + [g for gs in subtype_genes.values() for g in gs]) if g in da.var_names]
expr = sc.get.obs_df(da, keys=panel, layer="lognorm")
cl = da.obs["subcluster"].astype(str)
means = expr.groupby(cl.to_numpy()).mean()
zmeans = (means - means.mean()) / means.std(ddof=0).replace(0, 1)

rows = []
for c in zmeans.index:
    z = zmeans.loc[c]
    family = "SOX6" if z.get("SOX6", 0) - z.get("CALB1", 0) > 0 else "CALB1"
    cands = {s: np.mean([z[g] for g in gs[1:] if g in z.index]) for s, gs in subtype_genes.items()
             if gs[0] == family and any(g in z.index for g in gs[1:])}
    ranked = sorted(cands.items(), key=lambda kv: -kv[1])
    best, second = ranked[0], (ranked[1] if len(ranked) > 1 else (None, np.nan))
    rows.append({"subcluster": c, "cells": int((cl == c).sum()), "family": family,
                 "SOX6_minus_CALB1": round(z.get("SOX6", 0) - z.get("CALB1", 0), 2),
                 "subtype": best[0], "score": round(best[1], 2),
                 "runner_up": second[0], "runner_up_score": round(second[1], 2),
                 "margin": round(best[1] - second[1], 2)})
labels = pd.DataFrame(rows).set_index("subcluster").sort_values(["family", "subtype"])
da.obs["subtype"] = cl.map(labels["subtype"]).astype(str)
da.obs["group"] = np.where(da.obs["subtype"] == "SOX6_AGTR1", "vulnerable",
                   np.where(da.obs["subtype"].str.startswith("CALB1"), "resilient", "other_SOX6"))
pd.set_option("display.width", 250)
print(labels.to_string())
print("\nNeurons per subtype:")
print(da.obs["subtype"].value_counts().to_string())
print("\nCross-check against step 2 clusters (rows: subtype, columns: step 2 DA cluster):")
xt = pd.crosstab(da.obs["subtype"], da.obs["DA_cluster"])
print(xt.to_string())
labels.to_csv(f"{OUT}/tables/subcluster_subtype_labels.csv")
xt.to_csv(f"{OUT}/tables/subtype_vs_step2_cluster.csv")
order = [s for s in kamath_subtypes if s in set(da.obs["subtype"])]
sc.pl.dotplot(da, var_names=[g for g in kamath_panel if g in da.var_names], groupby="subtype",
              categories_order=order, layer="lognorm", standard_scale="var", show=False)
plt.savefig(f"{OUT}/figures/fig_subtype_marker_dotplot.png", dpi=200, bbox_inches="tight"); plt.show()
sc.pl.umap(da, color=["subtype", "SOX6", "CALB1", "AGTR1"], layer="lognorm", ncols=2, frameon=False, show=False)
plt.savefig(f"{OUT}/figures/fig_subtype_umap.png", dpi=200, bbox_inches="tight"); plt.show()


## Cell 7 — Score every neuron for senescence and aging

In [ ]:
# Score every neuron for senescence and aging (1-2 minutes)
marker_genes = set(kamath_panel) | {g for gs in subtype_genes.values() for g in gs}
score_sets = {}
for name in ["SenMayo", "CellAge_induces", "LU_AGING_BRAIN_UP"]:
    gs = gene_sets.get(name)
    if not gs:
        print(f"{name}: not available, skipped")
        continue
    kept = [g for g in gs if g in da.var_names and g not in marker_genes]
    removed = sorted(set(gs) & marker_genes)
    score_sets[name] = kept
    print(f"{name}: {len(kept)} genes used ({len(gs)} in list; {len(removed)} subtype markers removed: {removed})")
    sc.tl.score_genes(da, kept, score_name=name, use_raw=False, random_state=0)
for g in ["CDKN1A", "CDKN2A"]:
    if g in da.var_names:
        da.obs[g] = sc.get.obs_df(da, keys=[g], layer="lognorm")[g].to_numpy()
        print(f"{g}: detected in {(da.obs[g] > 0).mean():.1%} of neurons")
    else:
        print(f"{g}: not in the data")
READOUTS = [r for r in ["SenMayo", "CDKN1A", "CDKN2A", "CellAge_induces", "LU_AGING_BRAIN_UP"] if r in da.obs]
cols = ["donor", "sample", "status", "disease", "sex", "age", "facs", "subtype", "group",
        "n_genes_raw", "total_counts_raw"] + READOUTS
da.obs[cols].to_csv(f"{OUT}/tables/per_neuron_scores.csv")
print(f"\nReadouts: {READOUTS}. Per-neuron scores saved.")


## Cell 8 — H1, the main test
Compares vulnerable (SOX6_AGTR1) and resilient (CALB1+) neurons **inside each control donor**.
### 🛑 CHECKPOINT 3 — send Claude a screenshot of the result and the plot.

In [ ]:
# H1 (primary): vulnerable vs resilient neurons within each CONTROL donor (seconds)
def donor_pairs(obs, readout, min_cells, keep=None):
    d = obs if keep is None else obs[keep]
    d = d[d["group"].isin(["vulnerable", "resilient"])]
    n = d.groupby(["donor", "group"]).size().unstack(fill_value=0)
    m = d.groupby(["donor", "group"])[readout].mean().unstack()
    ok = n.index[(n.get("vulnerable", 0) >= min_cells) & (n.get("resilient", 0) >= min_cells)]
    out = m.loc[ok, ["vulnerable", "resilient"]].copy()
    out["diff"] = out["vulnerable"] - out["resilient"]
    out["n_vulnerable"], out["n_resilient"] = n.loc[ok, "vulnerable"], n.loc[ok, "resilient"]
    meta = d.groupby("donor")[["status", "disease", "sex", "age"]].first()
    return out.join(meta)

def paired_test(p):
    k = len(p)
    if k < 3:
        return {"donors": k, "mean_diff": np.nan, "ci_low": np.nan, "ci_high": np.nan, "t": np.nan,
                "p_ttest": np.nan, "cohens_dz": np.nan, "p_wilcoxon": np.nan}
    t = stats.ttest_1samp(p["diff"], 0.0)
    se = p["diff"].std(ddof=1) / np.sqrt(k)
    q = stats.t.ppf(0.975, k - 1)
    w = stats.wilcoxon(p["diff"]).pvalue if (p["diff"] != 0).any() else np.nan
    return {"donors": k, "mean_diff": p["diff"].mean(), "ci_low": p["diff"].mean() - q * se,
            "ci_high": p["diff"].mean() + q * se, "t": t.statistic, "p_ttest": t.pvalue,
            "cohens_dz": p["diff"].mean() / p["diff"].std(ddof=1), "p_wilcoxon": w}

def mixed_slope(obs, readout, donors):
    d = obs[obs["donor"].isin(donors) & obs["group"].isin(["vulnerable", "resilient"])].copy()
    d["vuln"] = (d["group"] == "vulnerable").astype(float)
    d["y"] = d[readout].astype(float)
    try:
        r = smf.mixedlm("y ~ vuln", d, groups=d["donor"], re_formula="~vuln").fit(reml=True, method="lbfgs")
        if not r.converged:
            raise RuntimeError("not converged")
        kind = "random slope"
    except Exception:
        r = smf.mixedlm("y ~ vuln", d, groups=d["donor"]).fit(reml=True)
        kind = "random intercept only (slope did not converge)"
    return {"mixed_beta": r.params["vuln"], "mixed_p": r.pvalues["vuln"], "mixed_model": kind}

ctrl = da.obs["status"] == "Ctrl"
MIN_CELLS = 20
pairs = donor_pairs(da.obs, "SenMayo", MIN_CELLS, keep=ctrl)
if len(pairs) < 5:
    print(f"Only {len(pairs)} control donors have >=20 neurons of each group; using the pre-specified fallback of 10.")
    MIN_CELLS = 10
    pairs = donor_pairs(da.obs, "SenMayo", MIN_CELLS, keep=ctrl)
h1 = paired_test(pairs)
h1.update(mixed_slope(da.obs, "SenMayo", pairs.index))
print(f"Control donors included: {len(pairs)} (minimum {MIN_CELLS} neurons per group)\n")
print(pairs.round(3).to_string())
print("\nH1 RESULT (SenMayo, controls, vulnerable minus resilient):")
for k, v in h1.items():
    print(f"  {k}: {v:.4g}" if isinstance(v, (float, np.floating)) else f"  {k}: {v}")
pairs.to_csv(f"{OUT}/tables/H1_donor_pairs_SenMayo.csv")
pd.Series(h1).to_csv(f"{OUT}/tables/H1_result_SenMayo.csv")

fig, ax = plt.subplots(figsize=(4, 4.5))
for d, r in pairs.iterrows():
    ax.plot([0, 1], [r["resilient"], r["vulnerable"]], "-o", color="grey", alpha=0.8)
    ax.annotate(d, (1.03, r["vulnerable"]), fontsize=7, va="center")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Resilient\n(CALB1+)", "Vulnerable\n(SOX6_AGTR1)"])
ax.set_ylabel("SenMayo score (donor mean)")
ax.set_title(f"H1, control donors (n={len(pairs)})\nmean diff {h1['mean_diff']:.3f}, paired t p={h1['p_ttest']:.3g}", fontsize=9)
ax.set_xlim(-0.3, 1.4)
plt.savefig(f"{OUT}/figures/fig_H1_SenMayo_paired.png", dpi=200, bbox_inches="tight"); plt.show()


## Cell 9 — H2 (disease), H3 (age) and the secondary readouts

In [ ]:
# H2 (disease) and H3 (age), plus the secondary readouts (seconds)
def h2_test(obs, readout, min_cells, keep=None):
    # The planned model, score ~ subtype * status + (1 | donor) on donor-by-subtype means, has exactly two
    # rows per donor. Its interaction term is then identical to comparing the per-donor differences between
    # diagnosis groups (diff ~ disease), which always converges, so we compute it that way.
    # Sex and age as main effects cancel out of a within-donor difference; the "adjusted" version below
    # adds them to the difference model, which is the stricter adjustment.
    p = donor_pairs(obs, readout, min_cells, keep=keep)
    p["dis"] = (p["disease"] == "Disease").astype(float)
    res = {"donors_ctrl": int((p["disease"] == "Ctrl").sum()), "donors_disease": int((p["disease"] == "Disease").sum())}
    for formula, tag in [("diff ~ dis", "interaction"), ("diff ~ dis + C(sex) + age", "interaction_adj")]:
        try:
            r = smf.ols(formula, p.dropna(subset=["diff", "age"])).fit()
            res.update({f"{tag}_beta": r.params["dis"], f"{tag}_p": r.pvalues["dis"]})
        except Exception as e:
            res.update({f"{tag}_beta": np.nan, f"{tag}_p": np.nan})
            print(f"  H2 {tag} for {readout} could not be fitted: {e}")
    return res, p

def h3_test(p):
    c = p[p["disease"] == "Ctrl"].dropna(subset=["age"])
    if len(c) < 4:
        return {"h3_rho": np.nan, "h3_p": np.nan, "h3_donors": len(c)}
    r = stats.spearmanr(c["age"], c["diff"])
    return {"h3_rho": r.correlation, "h3_p": r.pvalue, "h3_donors": len(c)}

results = []
for ro in READOUTS:
    pr = donor_pairs(da.obs, ro, MIN_CELLS, keep=ctrl)
    row = {"readout": ro, "role": "primary" if ro == "SenMayo" else "secondary"}
    row.update(paired_test(pr)); row.update(mixed_slope(da.obs, ro, pr.index))
    h2, pall = h2_test(da.obs, ro, MIN_CELLS); row.update(h2); row.update(h3_test(pall))
    results.append(row)
    pall.to_csv(f"{OUT}/tables/donor_pairs_all_{ro}.csv")
# CDKN1A and CDKN2A as donor-level pseudobulk: summed raw counts per donor and group, as log counts per million
def pseudobulk_pairs(gene, min_cells, keep=None):
    base = donor_pairs(da.obs, "total_counts_raw", min_cells, keep=keep)
    d = da.obs if keep is None else da.obs[keep]
    d = d[d["group"].isin(["vulnerable", "resilient"]) & d["donor"].isin(base.index)]
    gcounts = np.asarray(da[d.index, gene].layers["counts"].todense()).ravel() if sparse.issparse(da.layers["counts"]) \
        else np.asarray(da[d.index, gene].layers["counts"]).ravel()
    tmp = pd.DataFrame({"donor": d["donor"].to_numpy(), "group": d["group"].to_numpy(), "g": gcounts,
                        "tot": d["total_counts_raw"].to_numpy()})
    agg = tmp.groupby(["donor", "group"])[["g", "tot"]].sum()
    lcpm = np.log1p(agg["g"] / agg["tot"] * 1e6).unstack()
    out = base.copy()
    out["vulnerable"], out["resilient"] = lcpm.loc[base.index, "vulnerable"], lcpm.loc[base.index, "resilient"]
    out["diff"] = out["vulnerable"] - out["resilient"]
    return out
for g in ["CDKN1A", "CDKN2A"]:
    if g in da.var_names:
        pr = pseudobulk_pairs(g, MIN_CELLS, keep=ctrl)
        row = {"readout": f"{g}_pseudobulk", "role": "secondary"}
        row.update(paired_test(pr)); row.update(h3_test(pseudobulk_pairs(g, MIN_CELLS)))
        results.append(row)
        pr.to_csv(f"{OUT}/tables/donor_pairs_ctrl_{g}_pseudobulk.csv")
res = pd.DataFrame(results).set_index("readout")

# Benjamini-Hochberg across all secondary tests (H1 SenMayo is the single pre-specified primary test)
from statsmodels.stats.multitest import multipletests
pcols = ["p_ttest", "interaction_p", "interaction_adj_p", "h3_p"]
sec = [(ro, c) for ro in res.index for c in pcols if not (ro == "SenMayo" and c == "p_ttest") and pd.notna(res.loc[ro, c])]
if sec:
    q = multipletests([res.loc[ro, c] for ro, c in sec], method="fdr_bh")[1]
    for (ro, c), qq in zip(sec, q):
        res.loc[ro, c.replace("p_ttest", "q").replace("_p", "_q") if c != "p_ttest" else "q_ttest"] = qq
show = ["role", "donors", "mean_diff", "ci_low", "ci_high", "p_ttest", "q_ttest", "cohens_dz", "p_wilcoxon",
        "mixed_beta", "mixed_p", "donors_ctrl", "donors_disease", "interaction_beta", "interaction_p", "interaction_q",
        "interaction_adj_beta", "interaction_adj_p", "interaction_adj_q",
        "h3_rho", "h3_p", "h3_q"]
res = res.reindex(columns=[c for c in show if c in res.columns] + [c for c in res.columns if c not in show])
print(res[[c for c in show if c in res.columns]].round(4).T.to_string())
res.to_csv(f"{OUT}/tables/all_readouts_H1_H2_H3.csv")

pall = pd.read_csv(f"{OUT}/tables/donor_pairs_all_SenMayo.csv", index_col=0)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for k, (lab, sub) in enumerate(pall.groupby("disease")):
    ax[0].scatter(np.full(len(sub), k) + np.random.default_rng(0).uniform(-0.08, 0.08, len(sub)), sub["diff"], color="grey")
ax[0].axhline(0, color="black", lw=0.8); ax[0].set_xticks(range(pall["disease"].nunique()))
ax[0].set_xticklabels(sorted(pall["disease"].unique())); ax[0].set_ylabel("Vulnerable minus resilient (SenMayo)")
ax[0].set_title("H2: gap by diagnosis (one dot per donor)", fontsize=9)
c = pall[pall["disease"] == "Ctrl"]
ax[1].scatter(c["age"], c["diff"], color="grey"); ax[1].axhline(0, color="black", lw=0.8)
ax[1].set_xlabel("Donor age (years)"); ax[1].set_ylabel("Vulnerable minus resilient (SenMayo)")
ax[1].set_title("H3: gap vs age, control donors (exploratory)", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/figures/fig_H2_H3_SenMayo.png", dpi=200, bbox_inches="tight"); plt.show()


## Cell 10 — Sensitivity analyses
Re-runs the main test in different ways to check the result is robust.

In [ ]:
# Sensitivity analyses for H1 (1-3 minutes)
sens = []
def add(label, obs, readout="SenMayo", min_cells=MIN_CELLS, keep=None):
    p = donor_pairs(obs, readout, min_cells, keep=keep)
    r = paired_test(p); r["analysis"] = label; sens.append(r)

add("Primary (for reference)", da.obs, keep=ctrl)
add("Minimum 10 neurons", da.obs, min_cells=10, keep=ctrl)
add("Minimum 50 neurons", da.obs, min_cells=50, keep=ctrl)
add("FACS-positive libraries only", da.obs, keep=ctrl & (da.obs["facs"] == "Positive"))

# Depth adjustment: residualise the score on log total counts and genes detected
d = da.obs.copy()
print("Sequencing depth by group (control donors, median per neuron):")
print(d[ctrl & d["group"].isin(["vulnerable", "resilient"])].groupby("group")[["total_counts_raw", "n_genes_raw"]].median().round(0).to_string(), "\n")
X = np.column_stack([np.ones(len(d)), np.log1p(d["total_counts_raw"]), np.log1p(d["n_genes_raw"])])
beta, *_ = np.linalg.lstsq(X, d["SenMayo"].to_numpy(), rcond=None)
d["SenMayo_depth_adj"] = d["SenMayo"].to_numpy() - X @ beta
add("Depth-adjusted score", d, readout="SenMayo_depth_adj", keep=ctrl)

# AUCell-style score: area under the recovery curve of SenMayo genes within each neuron's top 5% genes
def aucell(adata, genes, top_frac=0.05, chunk=2000):
    Xl = adata.layers["lognorm"]; idx = np.array([adata.var_names.get_loc(g) for g in genes])
    k = max(1, int(top_frac * adata.n_vars)); out = np.zeros(adata.n_obs)
    gene_mask = np.zeros(adata.n_vars, bool); gene_mask[idx] = True
    max_auc = sum(k - i for i in range(min(len(idx), k)))
    for s in range(0, adata.n_obs, chunk):
        blk = Xl[s:s + chunk].toarray() if sparse.issparse(Xl) else Xl[s:s + chunk]
        top = np.argsort(-blk, axis=1, kind="stable")[:, :k]
        hit = gene_mask[top] & (np.take_along_axis(blk, top, axis=1) > 0)   # a gene not detected is never a hit
        out[s:s + chunk] = (hit * (k - np.arange(k))).sum(axis=1) / max_auc
    return out
d["SenMayo_AUCell"] = aucell(da, score_sets["SenMayo"])
add("AUCell scoring", d, readout="SenMayo_AUCell", keep=ctrl)

# High-confidence labels only (rule fixed at Checkpoint 2, before scoring): drop sub-clusters whose
# SOX6-vs-CALB1 family call is weak (|SOX6_minus_CALB1| < 0.5) and SOX6_AGTR1 sub-clusters whose lead over
# the next SOX6 subtype is small (margin < 0.25)
weak = labels.index[(labels["SOX6_minus_CALB1"].abs() < 0.5) |
                    ((labels["subtype"] == "SOX6_AGTR1") & (labels["margin"] < 0.25))]
print(f"Low-confidence sub-clusters left out of the high-confidence analysis: {list(weak)}")
add("High-confidence sub-clusters only", da.obs, keep=ctrl & ~da.obs["subcluster"].astype(str).isin(weak))

# Leave-one-donor-out
for drop in pairs.index:
    add(f"Leave out donor {drop}", da.obs, keep=ctrl & (da.obs["donor"] != drop))

# PD only, without LBD (applies to H2)
h2_pd, _ = h2_test(da.obs, "SenMayo", MIN_CELLS, keep=da.obs["status"] != "LBD")
sensdf = pd.DataFrame(sens).set_index("analysis")
print(sensdf[["donors", "mean_diff", "ci_low", "ci_high", "p_ttest", "cohens_dz", "p_wilcoxon"]].round(4).to_string())
print(f"\nH2 with PD only (LBD excluded): interaction beta {h2_pd['interaction_beta']:.4g}, p = {h2_pd['interaction_p']:.4g} "
      f"({h2_pd['donors_ctrl']} control, {h2_pd['donors_disease']} PD donors)")
sensdf.to_csv(f"{OUT}/tables/H1_sensitivity.csv")
pd.Series(h2_pd).to_csv(f"{OUT}/tables/H2_PD_only.csv")


## Cell 11 — Save everything to Drive
Send Claude a screenshot of the file list.

In [ ]:
# Save the labelled neurons and list everything saved (1-2 minutes)
keep_cols = [c for c in da.obs.columns]
da.obs[keep_cols].to_csv(f"{OUT}/tables/per_neuron_scores_full.csv")
out = da.copy(); out.X = out.layers["counts"]
for l in ["counts", "lognorm"]:
    del out.layers[l]
for k in ["X_pca", "X_pca_harmony"]:
    out.obsm.pop(k, None)
out.write_h5ad(f"{OUT}/GSE178265_DA_neurons_step3.h5ad", compression="gzip")
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f); print(f"{os.path.getsize(p) / 1e6:8.2f} MB  {p.replace(DRIVE + '/', '')}")
